# Week 6: Statistical Validation & Predictive Maintenance (PdM)
**Notebook:** `week6_stats_and_pdm.ipynb`

**Data Generation Strategy:** We will exclusively use Scikit-Learn's `make_*` functions (the "makers") to generate our synthetic operational and sensor data. This ensures reproducibility and best-practice data simulation.

This notebook covers:
1. **Statistical Validation** (T-test & Confidence Intervals) using `make_blobs`.
2. **PdM Feature Engineering** (Rolling stats, RMS, Range) using `make_regression`.
3. **Predictive Modeling** (Logistic Regression & Evaluation).

## 0. Imports & Global Settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import make_blobs, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_absolute_error, mean_squared_error

# Set style for better visualizations
sns.set_theme(style="whitegrid")
np.random.seed(42)  # For additional random operations (like adding spikes)

---
## Part 1: Statistical Validation (Using `make_blobs`)

**Business Hypothesis:** Machine A is faster (higher throughput) than Machine B.

- **Null Hypothesis (H₀):** Mean throughput of Machine A <= Machine B.
- **Alternative Hypothesis (H₁):** Mean throughput of Machine A > Machine B (One-tailed test).

We use `make_blobs` to generate two distinct "clusters" of throughput values centered around different means.

In [ ]:
# Generate synthetic operational dataset using make_blobs
# Machine A: average throughput of 105 units/hr, with a standard deviation of ~6
A_throughput, _ = make_blobs(n_samples=50, centers=[[105]], cluster_std=6, n_features=1, random_state=42)
A_throughput = A_throughput.flatten()

# Machine B: average throughput of 100 units/hr, with a standard deviation of ~5
B_throughput, _ = make_blobs(n_samples=50, centers=[[100]], cluster_std=5, n_features=1, random_state=123)
B_throughput = B_throughput.flatten()

# Combine into a single DataFrame for easier manipulation
df_A = pd.DataFrame({'Machine': 'A', 'Throughput': A_throughput})
df_B = pd.DataFrame({'Machine': 'B', 'Throughput': B_throughput})
df_ops = pd.concat([df_A, df_B], ignore_index=True)

print("### Operational Dataset Sample (First 5 rows):")
print(df_ops.head())
print(f"\nMachine A - Mean: {np.mean(A_throughput):.2f}, Std: {np.std(A_throughput):.2f}")
print(f"Machine B - Mean: {np.mean(B_throughput):.2f}, Std: {np.std(B_throughput):.2f}")

## 1.1 Perform Independent T-Test (One-Tailed)

In [ ]:
# Perform Levene's test for equal variance (to decide which t-test variant to use)
levene_stat, levene_p = stats.levene(A_throughput, B_throughput)
print(f"Levene's test for equal variance: p-value = {levene_p:.4f}")

if levene_p > 0.05:
    print("Variances are approximately equal. Using standard T-test.")
    t_stat, p_value = stats.ttest_ind(A_throughput, B_throughput, alternative='greater')
else:
    print("Variances are not equal. Using Welch's T-test.")
    t_stat, p_value = stats.ttest_ind(A_throughput, B_throughput, equal_var=False, alternative='greater')

print("\n--- T-Test Results ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value (One-tailed): {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\n✅ Result: p-value ({p_value:.4f}) < {alpha}. Reject the Null Hypothesis.")
    print("Conclusion: There is statistically significant evidence that Machine A is faster than Machine B.")
else:
    print(f"\n❌ Result: p-value ({p_value:.4f}) >= {alpha}. Fail to reject the Null Hypothesis.")

## 1.2 Calculate 95% Confidence Intervals for Key Metrics

In [ ]:
def calculate_ci(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    std_err = stats.sem(data)
    margin = std_err * stats.t.ppf((1 + confidence) / 2, n - 1)
    return mean, (mean - margin, mean + margin)

mean_a, ci_a = calculate_ci(A_throughput)
mean_b, ci_b = calculate_ci(B_throughput)

print("### 95% Confidence Intervals for Throughput:")
print(f"Machine A: Mean = {mean_a:.2f}, 95% CI = ({ci_a[0]:.2f}, {ci_a[1]:.2f})")
print(f"Machine B: Mean = {mean_b:.2f}, 95% CI = ({ci_b[0]:.2f}, {ci_b[1]:.2f})")

# Confidence Interval for the Difference in Means
mean_diff = mean_a - mean_b
se_diff = np.sqrt( (np.var(A_throughput, ddof=1)/len(A_throughput)) + (np.var(B_throughput, ddof=1)/len(B_throughput)) )
df_diff = ( (np.var(A_throughput, ddof=1)/len(A_throughput) + np.var(B_throughput, ddof=1)/len(B_throughput))**2 ) / ( ( (np.var(A_throughput, ddof=1)/len(A_throughput))**2 / (len(A_throughput)-1) ) + ( (np.var(B_throughput, ddof=1)/len(B_throughput))**2 / (len(B_throughput)-1) ) )
ci_diff_lower = mean_diff - stats.t.ppf(0.975, df_diff) * se_diff
ci_diff_upper = mean_diff + stats.t.ppf(0.975, df_diff) * se_diff

print(f"\nDifference in Means (A - B): {mean_diff:.2f}")
print(f"95% CI for the difference: ({ci_diff_lower:.2f}, {ci_diff_upper:.2f})")
print("✅ Interpretation: Since this interval is entirely above 0, we are 95% confident Machine A truly outperforms B.")

---
## Part 2: PdM Feature Engineering (Using `make_regression`)

**Task:** Simulate a bearing's vibration signal degrading over time.

We use `make_regression` to generate a strong linear upward trend, which perfectly mimics the gradual increase in vibration amplitude as a bearing wears out.

In [ ]:
# Generate synthetic vibration data using make_regression
n_samples = 500
time_axis = np.arange(n_samples).reshape(-1, 1)  # Feature: Time

# Generate a target variable (vibration) with a strong linear slope and some inherent noise
# We'll generate a pure linear trend and scale it to a realistic vibration range (2.0 to 12.0 mm/s)
vibration_trend, _ = make_regression(n_samples=n_samples, n_features=1, noise=0.0, random_state=456)

# Min-Max scale the trend to fit between 2.0 and 12.0
vibration_trend = 2.0 + 10.0 * (vibration_trend - vibration_trend.min()) / (vibration_trend.max() - vibration_trend.min())
vibration_trend = vibration_trend.flatten()

# Add realistic Gaussian noise (sensor jitter)
noise = np.random.normal(0, 0.5, n_samples)

# Add occasional random impact spikes (simulating debris passing through)
spike_indices = np.random.choice(n_samples, size=15, replace=False)
spikes = np.zeros(n_samples)
spikes[spike_indices] = np.random.uniform(2, 5, size=15)

# Final vibration signal = Trend + Noise + Spikes
vibration_signal = vibration_trend + noise + spikes
vibration_signal = np.maximum(vibration_signal, 0.1)  # Floor at 0.1

df_sensor = pd.DataFrame({
    'Timestamp': time_axis.flatten(),
    'Vibration_Amplitude': vibration_signal
})

print("### Sensor Dataset Sample (First 5 rows):")
print(df_sensor.head())

## 2.1 Engineer 3 Health Features
1. **Rolling Mean (SMA)** - Smooths out spikes to show the underlying degradation trend.
2. **Root Mean Square (RMS)** - Measures the average energy of the vibration over a window.
3. **Rolling Range (Max - Min)** - Captures the increasing severity of peak-to-peak fluctuations.

In [ ]:
window_size = 20

# Feature 1: Rolling Mean
df_sensor['Rolling_Mean'] = df_sensor['Vibration_Amplitude'].rolling(window=window_size, min_periods=1).mean()

# Feature 2: Rolling RMS
df_sensor['Rolling_RMS'] = df_sensor['Vibration_Amplitude'].rolling(window=window_size, min_periods=1).apply(
    lambda x: np.sqrt(np.mean(x**2)), raw=True
)

# Feature 3: Rolling Range (Max - Min)
df_sensor['Rolling_Range'] = df_sensor['Vibration_Amplitude'].rolling(window=window_size, min_periods=1).apply(
    lambda x: np.max(x) - np.min(x), raw=True
)

# Drop NaN rows caused by the rolling window (min_periods=1 prevents NaNs, but we drop first few for consistency)
df_sensor = df_sensor.dropna().reset_index(drop=True)

print("\n### Engineered Features Sample:")
print(df_sensor.head())

## 2.2 Visualize Features Against Time (Degradation Trends)
The plots clearly show an upward trajectory, perfectly modeling the physical reality of bearing wear.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Plot 1: Raw vs Rolling Mean
axes[0].plot(df_sensor['Timestamp'], df_sensor['Vibration_Amplitude'], 
             label='Raw Signal', alpha=0.4, linewidth=0.8, color='gray')
axes[0].plot(df_sensor['Timestamp'], df_sensor['Rolling_Mean'], 
             label='Rolling Mean (Trend)', linewidth=2.5, color='red')
axes[0].set_ylabel('Amplitude (mm/s)')
axes[0].set_title('Feature 1: Rolling Mean - Shows the Overall Degradation Trend')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: RMS
axes[1].plot(df_sensor['Timestamp'], df_sensor['Rolling_RMS'], 
             label='Rolling RMS', linewidth=2, color='blue')
axes[1].set_ylabel('Energy (mm/s)')
axes[1].set_title('Feature 2: Rolling RMS - Captures Increasing Energy of Vibration')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Max-Min Range
axes[2].plot(df_sensor['Timestamp'], df_sensor['Rolling_Range'], 
             label='Rolling Range', linewidth=2, color='green')
axes[2].set_xlabel('Time (Minutes of Operation)')
axes[2].set_ylabel('Amplitude Range (mm/s)')
axes[2].set_title('Feature 3: Rolling Range - Captures Increasing Peak Severity')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2.3 Create a Binary Target Variable ("Healthy" vs. "Failing")
We set a threshold on the Rolling Mean. When the average vibration exceeds 7.5 mm/s, 
the bearing is considered "Failing".

In [ ]:
degradation_threshold = 7.5
df_sensor['Target'] = (df_sensor['Rolling_Mean'] > degradation_threshold).astype(int)

print("### Target Variable Distribution:")
print(df_sensor['Target'].value_counts())
print(f"\nHealthy (0): { (df_sensor['Target'] == 0).sum() } samples")
print(f"Failing (1): { (df_sensor['Target'] == 1).sum() } samples")

# Visual verification of the split
plt.figure(figsize=(12, 4))
plt.plot(df_sensor['Timestamp'], df_sensor['Rolling_Mean'], label='Rolling Mean', color='blue')
plt.axhline(y=degradation_threshold, color='red', linestyle='--', label='Failure Threshold')
plt.scatter(df_sensor[df_sensor['Target']==1]['Timestamp'], 
            df_sensor[df_sensor['Target']==1]['Rolling_Mean'], 
            color='red', s=30, label='Labeled as Failing', alpha=0.7)
plt.title('Binary Target Assignment (Healthy vs Failing)')
plt.xlabel('Time')
plt.ylabel('Vibration Amplitude')
plt.legend()
plt.show()

---
## Part 3: Predictive Modeling
**Goal:** Build a Logistic Regression model to predict failure (Target = 1) using the 3 engineered features.

The idea is to predict *early* failure so maintenance can be scheduled proactively.

In [ ]:
# Select Features (X) and Target (y)
features = ['Rolling_Mean', 'Rolling_RMS', 'Rolling_Range']
X = df_sensor[features]
y = df_sensor['Target']

# Train/Test Split (80/20) - we stratify to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize features (crucial for Logistic Regression to converge properly)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Build the model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]  # Probability of failure

## 3.1 Evaluate Model Performance

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"### Model Accuracy: {accuracy:.4f} ({(accuracy*100):.2f}%)")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\n### Confusion Matrix:")
print(pd.DataFrame(cm, columns=['Predicted 0 (Healthy)', 'Predicted 1 (Failing)'], 
                   index=['Actual 0 (Healthy)', 'Actual 1 (Failing)']))

# For Regression style metrics (if we want to evaluate probability error)
mae = mean_absolute_error(y_test, y_pred_proba)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_proba))
print(f"\n### Probability Evaluation Metrics (for the 'Failing' class):")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"RMSE (Root Mean Square Error): {rmse:.4f}")

print("\n### Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Healthy', 'Failing']))

## 3.2 Business Implication of the Model's Performance

In [ ]:
# Feature importance visualization
coefficients = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(coefficients['Feature'], coefficients['Coefficient'], color='skyblue')
plt.title('Logistic Regression Feature Coefficients (Larger = Higher Impact on Failure)')
plt.axvline(0, color='black', linestyle='-', linewidth=0.8)
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()